# NB 03: SSL Pretraining with Tuned Architecture

Re-run SSL pretraining (mask ratios 0.15, 0.30, 0.50) using the best
hyperparameters from the sweep, then fine-tune on all outcomes.

**Prerequisites:**
1. Run `00_setup_and_preprocessing.ipynb` first (saves artifact to Drive)
2. Run `02_hp_sweep.ipynb` first (saves sweep results)
3. Runtime -> Change runtime type -> **T4 GPU** (or A100)
4. GitHub PAT stored in Colab Secrets as `GITHUB_PAT`

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from google.colab import userdata
import os

# Retrieve the GitHub token from Colab Secrets
github_token = userdata.get('GITHUB_PAT')

username = "elroy-galbraith"
repository = "fin-jepa"
repo_url = f"https://{github_token}@github.com/{username}/{repository}.git"

if not os.path.exists(f'/content/{repository}'):
    !git clone {repo_url} /content/{repository}

%cd /content/{repository}

# Install fin-jepa as editable WITHOUT upgrading Colab's pre-installed deps
!pip install -q --no-deps -e '.[dev]'

# Install only the deps Colab doesn't already ship
!pip install -q hydra-core omegaconf mlflow optuna xgboost yfinance rich python-dotenv tqdm

In [ ]:
# Symlink data from Google Drive so relative paths work
import os

DRIVE_DATA = '/content/drive/MyDrive/Fin-JEPA/data'

for subdir in ['raw', 'processed']:
    target = f'data/{subdir}'
    if os.path.islink(target) or os.path.isdir(target):
        !rm -rf {target}
    os.symlink(f'{DRIVE_DATA}/{subdir}', target)

# Verify data exists
!ls -la data/raw/xbrl_features.parquet
!ls -la data/processed/label_database.parquet
!ls -la data/raw/market/market_aligned.parquet
print('Data linked successfully!')

# Verify GPU
import torch
assert torch.cuda.is_available(), 'No GPU detected! Change runtime to T4 or A100.'
print(f'GPU: {torch.cuda.get_device_name(0)}')
props = torch.cuda.get_device_properties(0)
print(f'VRAM: {props.total_memory / 1e9:.1f} GB')

In [ ]:
# Common setup: logging, configs, control flag
import json
import logging
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from omegaconf import OmegaConf

logging.basicConfig(level=logging.INFO, format='%(levelname)s %(name)s: %(message)s')

# Set to True to re-run experiments even if cached results exist
FORCE_RERUN = False

# Load configs from YAML (override individual values below if needed)
benchmark_cfg = OmegaConf.load('configs/study0/benchmark.yaml')
ssl_cfg = OmegaConf.load('configs/study0/ssl_experiment.yaml')

# Paths
RESULTS_DIR = Path('results/study0')
BENCHMARK_PATH = RESULTS_DIR / 'benchmark_results.json'
SWEEP_PATH = RESULTS_DIR / 'ft_sweep_results.json'
SSL_V2_PATH = RESULTS_DIR / 'ssl_experiment_results_v2.json'
FINAL_PATH = RESULTS_DIR / 'final_benchmark.json'

OUTCOMES = ['stock_decline', 'earnings_restate', 'audit_qualification',
            'sec_enforcement', 'bankruptcy']

print(f'Results dir: {RESULTS_DIR}')
print(f'Benchmark cached: {BENCHMARK_PATH.exists()}')
print(f'Sweep cached: {SWEEP_PATH.exists()}')
print(f'SSL v2 cached: {SSL_V2_PATH.exists()}')
print(f'FORCE_RERUN: {FORCE_RERUN}')

# Preprocessing artifact path (shared across split notebooks)
ARTIFACT_DIR = Path('/content/drive/MyDrive/Fin-JEPA/artifacts/study0')
ARTIFACT_PATH = ARTIFACT_DIR / 'preprocessing_v1.pkl'

SEED = int(OmegaConf.select(benchmark_cfg, 'training.seed', default=42))

mask_ratios = [0.15, 0.30, 0.50]
MIN_POSITIVES = 20


In [ ]:
import pickle

if not ARTIFACT_PATH.exists():
    raise FileNotFoundError(
        f'Preprocessing artifact not found at {ARTIFACT_PATH}.\n'
        'Run 00_setup_and_preprocessing.ipynb first.'
    )

with open(ARTIFACT_PATH, 'rb') as f:
    _artifact = pickle.load(f)

splits = _artifact['splits']
scaler = _artifact['scaler']
feature_cols = _artifact['feature_cols']
categorical_cols = _artifact['categorical_cols']
n_features = _artifact['n_features']
n_cat = _artifact['n_cat']
cat_cards = _artifact['cat_cards']
config_fingerprint = _artifact['config_fingerprint']

print(f'Loaded preprocessing artifact (fingerprint: {config_fingerprint})')
print(f'  Train: {len(splits["train"]):,} | Val: {len(splits["val"]):,} | Test: {len(splits["test"]):,}')
print(f'  Features: {n_features} ({n_cat} categorical)')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [ ]:
SWEEP_PATH = RESULTS_DIR / 'ft_sweep_results.json'
if not SWEEP_PATH.exists():
    raise FileNotFoundError(
        f'Sweep results not found at {SWEEP_PATH}.\n'
        'Run 02_hp_sweep.ipynb first.'
    )

with open(SWEEP_PATH) as f:
    sweep_results = json.load(f)

best_cfg = sweep_results['best_config']
print(f"Best sweep config: d_token={best_cfg['d_token']}, n_layers={best_cfg['n_layers']}, "
      f"n_heads={best_cfg['n_heads']}, lr={best_cfg['lr']}, mean_auroc={best_cfg['mean_auroc']:.4f}")

## 4. SSL Pretraining with Tuned Architecture

Re-run SSL pretraining (mask ratios 0.15, 0.30, 0.50) using the best
hyperparameters from the sweep, then fine-tune on all outcomes.

In [ ]:
%%time
if not FORCE_RERUN and SSL_V2_PATH.exists():
    print(f'Loading cached SSL v2 results from {SSL_V2_PATH}')
    with open(SSL_V2_PATH) as f:
        ssl_results = json.load(f)
else:
    # Build tuned SSL config from best sweep params
    best = sweep_results['best_config']
    ssl_overrides = {
        'ft_transformer': {
            'd_token': best['d_token'],
            'n_heads': best['n_heads'],
            'n_layers': best['n_layers'],
        },
        # ATS-217: use sweep-winning lr for fine-tuning so the SSL
        # scratch baseline matches Cell 24's final_benchmark ft_scratch.
        'training': {'learning_rate': best['lr']},
        'ssl_experiment': {'force_pretrain': True},
        'checkpoint_dir': 'models/checkpoints/study0_ssl_experiment_v2',
    }
    ssl_cfg_v2 = OmegaConf.merge(ssl_cfg, ssl_overrides)

    from fin_jepa.training.pretrain_ssl import run_ssl_experiment
    # ATS-217: pass the shared preprocessing pipeline so the SSL scratch
    # baseline is identical to the main benchmark (Tables 3/5).
    ssl_results = run_ssl_experiment(
        ssl_cfg_v2,
        prebuilt_splits=splits,
        prebuilt_feature_cols=feature_cols,
        prebuilt_cat_cols=categorical_cols,
    )

    # Save as v2
    RESULTS_DIR.mkdir(parents=True, exist_ok=True)
    with open(SSL_V2_PATH, 'w') as f:
        json.dump(ssl_results, f, indent=2, default=str)
    print('SSL v2 experiment complete.')

print(f"\nSSL Recommendation: {ssl_results['recommendation']}")
print(f"Architecture: d_token={best_cfg['d_token']}, n_layers={best_cfg['n_layers']}, "
      f"n_heads={best_cfg['n_heads']}")

In [ ]:
# SSL v2 results: AUROC sweep table + all 4 metrics for scratch vs SSL mr=0.15
mask_ratios = sorted(ssl_results.get('pretrained', {}).keys())
FOUR_METRICS = ['auroc', 'auprc', 'brier', 'ece']
DEFAULT_MR = '0.15'

# AUROC across mask ratios
rows = []
for outcome in OUTCOMES:
    base_rec = (ssl_results.get('baseline') or {}).get(outcome) or {}
    baseline_auroc = base_rec.get('auroc')
    best_auroc, best_mr = baseline_auroc, 'none'
    for mr in mask_ratios:
        auroc = (ssl_results['pretrained'].get(mr) or {}).get(outcome, {}).get('auroc')
        if auroc is not None and (best_auroc is None or auroc > best_auroc):
            best_auroc = auroc
            best_mr = mr
    delta = (best_auroc - baseline_auroc) if baseline_auroc and best_auroc else None
    row = {'Outcome': outcome, 'FT tuned (scratch)': baseline_auroc}
    for mr in mask_ratios:
        row[f'mr={mr}'] = (ssl_results['pretrained'].get(mr) or {}).get(outcome, {}).get('auroc')
    row['Best MR'] = best_mr
    row['AUROC delta'] = delta
    rows.append(row)

ssl_auroc_df = pd.DataFrame(rows).set_index('Outcome')
print('=== AUROC across SSL mask ratios ===')
display(ssl_auroc_df.style.format('{:.4f}', na_rep='--',
    subset=[c for c in ssl_auroc_df.columns if c != 'Best MR']))

# Full 4-metric table: scratch vs SSL mr=DEFAULT_MR
rows2 = []
for outcome in OUTCOMES:
    base_rec = (ssl_results.get('baseline') or {}).get(outcome) or {}
    ssl_rec  = ((ssl_results.get('pretrained') or {}).get(DEFAULT_MR) or {}).get(outcome) or {}
    for metric in FOUR_METRICS:
        rows2.append({
            'Outcome': outcome,
            'Metric': metric.upper(),
            'FT scratch': base_rec.get(metric),
            f'FT SSL mr={DEFAULT_MR}': ssl_rec.get(metric),
        })
ssl_metrics_df = pd.DataFrame(rows2).set_index(['Outcome', 'Metric'])
print(f'\n=== All 4 metrics: scratch vs SSL mr={DEFAULT_MR} ===')
display(ssl_metrics_df.style.format('{:.4f}', na_rep='--'))


In [ ]:
# SSL pretraining loss curves
loss_curves = ssl_results.get('loss_curves', {})

if any(len(v) > 0 for v in loss_curves.values()):
    fig, ax = plt.subplots(figsize=(10, 4))
    for mr, losses in sorted(loss_curves.items()):
        if losses:
            ax.plot(range(1, len(losses) + 1), losses, label=f'mr={mr}')
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Reconstruction Loss (MSE)')
    ax.set_title('SSL Pretraining Loss Curves (Tuned Architecture)')
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    fig_dir = RESULTS_DIR / 'figures'
    fig_dir.mkdir(exist_ok=True)
    plt.savefig(fig_dir / 'ssl_loss_curves.png', bbox_inches='tight', dpi=300)
    plt.show()
else:
    print('No loss curves available (loaded from cache).')
